# Análisis de Inteligencia de Clientes: E-commerce de Mascotas 🐾

## Contexto de Negocio
Este proyecto se sitúa en una empresa de *e-commerce* especializada en productos para mascotas. Ante el crecimiento del volumen de ventas, el equipo de Producto identificó la necesidad de explotar las reseñas de los clientes para convertirlas en *insights* accionables.

El objetivo es transformar datos no estructurados (un archivo de texto plano con formato clave-valor) en una base de conocimiento que permita optimizar el catálogo de productos y mejorar la experiencia del usuario.

## Objetivos Estratégicos
* **Ingeniería de Datos:** Desarrollar un *parser* robusto para estructurar y limpiar un dataset de reseñas de e-commerce.
* **Detección de Riesgos:** Identificar productos de alta visibilidad (muchas reseñas) pero baja satisfacción para intervención inmediata del equipo de Producto.
* **Análisis de Tendencias:** Evaluar la evolución temporal de la satisfacción del cliente segmentada por líneas de negocio (rangos de precio).
* **Minería de Texto:** Comparar el lenguaje y patrones de palabras entre clientes satisfechos e insatisfechos.

---
## Flujo de Trabajo (Pipeline)

1. **Extracción y Estructuración:** Parseo de archivos `.txt` nativos a estructuras de datos relacionales.
2. **Preprocesamiento:** Limpieza de strings, normalización de tipos de datos y manejo de valores nulos en precios y ratings.
3. **Análisis Estadístico:** Agrupación de métricas de rendimiento por producto y usuario.
4. **Visualización de Datos:** Generación de reportes gráficos para identificar patrones temporales y lingüísticos.

## Resultados y Entregables
El análisis se consolida en la carpeta `/resultados`, donde se presentan:
* **Identificación de Productos Críticos:** Listado de ítems con baja satisfacción a pesar de su alto volumen de ventas.
* **Análisis de Sentimiento Visual:** *Wordclouds* comparativos de reseñas positivas vs. negativas.
* **Reporte de Tendencias:** Evolución de la experiencia del cliente a lo largo del tiempo segmentada por niveles de precio.

## Parte 1 — Preparación de los datos
1. Lectura de datos: Leer el archivo de texto y parsear cada reseña en un diccionario, separando las claves y valores de cada línea.

In [ ]:
#Importamos las librerias
#!pip install wordcloud
import pandas as pd
import matplotlib.pyplot as plt
import re
from wordcloud import STOPWORDS
from wordcloud import WordCloud

In [ ]:
#Leemos y guardamos el contenido del archivo .txt
with open("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Estadística para Ciencia de Datos/Tareas/Pet_Supplies.txt", "r") as f:
    contenido = f.read()
print(contenido)

In [ ]:
reseñas = contenido.split("\n\n") #creamos una lista que contiene cada reseña
print(len(reseñas)) #chequeamos largo de la lista (cantidad reseñas)


In [ ]:
#creamos una lista de diccionarios donde cada diccionario guarda la información de cada reseña en particular
lista_diccionarios = []

for reseña in reseñas:
    lineas = reseña.split("\n")
    diccionario = {}
    for texto in lineas:
        if ":" not in texto:
            continue
        clave_valor = texto.split(":",1)
        clave = clave_valor[0].strip()
        valor = clave_valor[1].strip()
        diccionario[clave] = valor
    lista_diccionarios.append(diccionario)
  

In [ ]:
print(len(lista_diccionarios)) #chequeo cantidad de diccionarios coincida con cantidad reseñas
print(lista_diccionarios[2])

## 2. Limpieza de datos: Preparar los datos para el análisis:

In [ ]:
df_reseñas = pd.DataFrame(lista_diccionarios) #convertimos la lista de diccionarios en un dataframe

In [ ]:
df_reseñas.tail()

### 2.1 Manejo de valores faltantes

In [ ]:
df_reseñas.isna().sum() #chequeamos vacíos

In [ ]:
df_reseñas[df_reseñas.isna().all(axis=1)] #comprobamos que la última fila es completamente null

In [ ]:
df_reseñas = df_reseñas.dropna(how="all") #eliminamos la fila con nulls

In [ ]:
df_reseñas.shape

In [ ]:
df_reseñas.describe(include="all")

In [ ]:
#Ajusto el nombre de las columnas para facil acceso
df_reseñas.columns = df_reseñas.columns.str.replace('/', '_')

Comentarios:

Observamos que hay valores del tipo "unknown" en varias columnas, vamos a replazarlos por vacío

In [ ]:
df_reseñas = df_reseñas.replace("unknown", pd.NA)

In [ ]:
df_reseñas.info() #ahora podemos visualizar la cantidad de datos faltantes por columna. Notamos que las columnas son todas tipo Object.

In [ ]:
df_reseñas.isna().sum()

In [ ]:
#Evaluamos eliminar datos faltantes según su peso en el total de la muestra.
print(f"% de datos faltantes: \n\n product_price: {df_reseñas["product_price"].isna().sum() / len(df_reseñas):.2%} \n review_userId: {df_reseñas["review_userId"].isna().sum() / len(df_reseñas):.2%} \n revier_profileName: {df_reseñas["review_profileName"].isna().sum() / len(df_reseñas):.2%}")

In [ ]:
#Elimino filas con datos faltantes en usuarios porque representan menos del 1% del total.

df_reseñas.dropna(subset=["review_userId","review_profileName"], inplace=True)

In [ ]:
df_reseñas.isna().sum()

In [ ]:
#Un producto puede tener varios precios?
df_reseñas.groupby("product_productId")["product_price"].value_counts().sort_values(ascending=False)

In [ ]:
ids=df_reseñas.groupby("product_productId")["product_price"].unique()
ids

In [ ]:
df_reseñas.groupby("product_productId")["product_price"] \
  .unique() \
  .loc[lambda x: x.apply(len) > 1]

### 2.2 Conversión del tipo de datos

In [ ]:
#Ajusto columnas a string

columnas_str = ['product_productId', 'product_title', 'review_userId',
       'review_profileName', 'review_helpfulness', 'review_summary', 'review_text']

for columna in columnas_str:
    df_reseñas[columna] = df_reseñas[columna].astype("string")

In [ ]:
#Reviso que valores toman la columna de precio y score
print(df_reseñas["product_price"].unique(), df_reseñas["review_score"].unique())

In [ ]:
#Ajusto columnas numéricas

df_reseñas["product_price"] = pd.to_numeric(df_reseñas["product_price"], errors = "coerce", downcast = "float")
df_reseñas["product_price"] = df_reseñas["product_price"].round(1)
df_reseñas["review_score"] = pd.to_numeric(df_reseñas["review_score"], errors = "coerce", downcast = "integer")


In [ ]:
#Seguimos con la misma cantidad de datos faltantes. Hay productos que nunca tienen precio informado
df_reseñas.isna().sum()

In [ ]:
#Hay reseñas donde el precio del producto fue 0
df_reseñas[df_reseñas["product_price"]==0].count()

In [ ]:
#Elimino las filas con precios 0
df_reseñas = df_reseñas[df_reseñas["product_price"]!=0]

In [ ]:
df_reseñas["product_price"].sort_values()

In [ ]:
#Hago un boxplot para ver si la variable precio tiene muchos valores atípicos, en ese caso uso la mediana para completar vacíos
fig, ax = plt.subplots(figsize=(8,6))

data = df_reseñas["product_price"].dropna()

boxplot = ax.boxplot(
    data,
    patch_artist=True,
    showfliers=True,
    flierprops=dict(marker='o', markerfacecolor='red', markersize=3, alpha=0.5)
)

plt.setp(boxplot['boxes'], color='lightblue')
plt.setp(boxplot['whiskers'], color='blue')
plt.setp(boxplot['caps'], color='gold')
plt.setp(boxplot['medians'], color='red')

ax.set_xlabel("Precio del producto")
ax.set_ylabel("Precio")
ax.set_title("Distribución de precios con outliers visibles")

plt.show()

In [ ]:
df_reseñas["product_price"].describe()

Comentarios: 
- La variable precio tiene una considerable cantidad de outliers, remplazo con mediana los vacíos, no uso promedio porque se vería demasiado sesgado ni elimino porque representan el 11% de la muestra.
- De hecho, incluso el 75% de los datos está por debajo del precio 34, mientras que el precio máximo es 600.
- También elimino los precios iguales a 0, interpreto que hubo algun error y son solo 9 observaciones.

In [ ]:
#Remplazo valores faltantes con la mediana
df_reseñas["product_price"] = df_reseñas["product_price"].fillna(df_reseñas["product_price"].median())

In [ ]:
df_reseñas["product_price"].describe()

In [ ]:
#Ajusto la fecha como datetime
df_reseñas["review_time"] = pd.to_datetime(df_reseñas["review_time"], unit="s")

In [ ]:
df_reseñas["review_time"].head()

In [ ]:
df_reseñas.info() #chequeo que el tipo de dato en cada columna sea el correcto

Comentario: el tipo de dato para cada columna ahora es el deseado.

In [ ]:
df_reseñas.describe(include="all")

In [ ]:
df_reseñas[["review_summary", "review_text"]].head()

In [ ]:
#Hacemos la limpieza del texto de las reseñas eliminando signos de puntuación u otros caracteres no deseados


columnas = ["review_summary", "review_text"]

for columna in columnas:
    df_reseñas[columna] = (
        df_reseñas[columna]
        .str.lower() #paso todo a minuscula
        .str.replace(r"[^\w\s']", " ", regex=True) #remplazo por todo lo que no es letras, numeros, espacios
        .str.replace(r"\s+", " ", regex=True) #reemplazo varios espacios por uno solo
        .str.strip()
    )

In [ ]:
df_reseñas[["review_summary", "review_text"]].head()

## Parte 2 — Análisis exploratorio

### 1. ¿Cuáles son los productos más populares y cómo los evalúan los clientes?

In [ ]:
df_reseñas.columns

In [ ]:
df_resumen = df_reseñas.groupby("product_title").agg(
    cantidad_reseñas=("review_score","count"), 
    promedio_score=("review_score","mean")).round(2).sort_values(by= "cantidad_reseñas", ascending=False)

df_resumen.head(10)



Comentarios:
- El producto más popular es FURminator deShedding Tool porque tiene 4.725 reviews.
- Dentro de los productos más populares, el primero en la tabla es también el de mayor score.
- A primera vista parece que productos populares tienen buen score, lo que suena lógico.

#### Exportamos este resultado a un csv separado

In [ ]:
df_resumen.to_csv("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Estadística para Ciencia de Datos/Tareas/resultados/2.1score.csv", index=False, encoding="utf-8")

### 2. ¿Existen productos con muchas reseñas pero baja satisfacción?

In [ ]:
productos_revision = (df_resumen["cantidad_reseñas"]>=50) & (df_resumen["promedio_score"]<3)

In [ ]:
df_productos_revision = df_resumen[productos_revision]

In [ ]:
len(df_productos_revision)

In [ ]:
df_productos_revision.sort_values(by= ["cantidad_reseñas", "promedio_score"], ascending=False)

Comentarios:
- Hay 30 productos candidatos que el equipo debería revisar. Son productos con una cantidad de reseñas considerable donde el score es bajo según el criterio


In [ ]:
df_productos_revision.to_csv("C:/Users/gonza/OneDrive/Escritorio/Master CD/Programacion DS/proyecto_final/2.2productos_revision.csv", index=False, encoding="utf-8")

### 3. ¿La satisfacción de los clientes ha cambiado con el tiempo?

Objetivo: ver la evolución del score en el tiempo según segmento de precio (bajo, medio, alto, muy alto)

In [ ]:
#Uso percentiles como criterio para establecer las categorias
df_reseñas["segmento_precio"], bins = pd.qcut(df_reseñas["product_price"], q=3, labels= ["bajo", "medio", "alto"], retbins=True)

In [ ]:
#Considerando la distribución de precios, estoy de acuerdo con utilizar estos bins, el último bin es amplio y contiene valores atípicos.
bins.round(0)

In [ ]:
df_reseñas[["product_price", "segmento_precio"]]

In [ ]:
#Creo variable con periodo trimestral desde la variable fecha
df_reseñas["trimestre"] = df_reseñas["review_time"].dt.to_period(freq= "Q")
df_reseñas["trimestre"].head()

In [ ]:
df_plot= df_reseñas.groupby(["trimestre","segmento_precio"])["review_score"].mean()
df_plot.head()

In [ ]:
df_plot = df_plot.reset_index()

In [ ]:
df_pivot = df_plot.pivot(
    index="trimestre",
    columns="segmento_precio",
    values="review_score"
)

In [ ]:
df_pivot.head()

In [ ]:
df_pivot.plot(
    figsize=(10,6),
    marker="o",
    linewidth=2,
    colormap="tab10"
)

plt.grid(linestyle="--", alpha=0.5)
plt.title("Evolución del score por segmento de precio", fontsize=14, fontweight= "bold")
plt.ylabel("Score promedio")
plt.xticks(rotation=45)
plt.legend(title="Segmento")

plt.show()

In [ ]:
#Hay pocos valores para los primeros años en la historia de precios y eso genera ruido en su evolución, voy a considerar información a partir del 2008.
df_reseñas["trimestre"].value_counts().sort_index(ascending=False)

In [ ]:
df_pivot = df_pivot.loc["2008":]

In [ ]:
pivot_png=df_pivot.plot(
    figsize=(10,6),
    marker="o",
    linewidth=2,
    colormap="tab10"
)

plt.grid(linestyle="--", alpha=0.5)
plt.title("Evolución del score por segmento de precio", fontsize=14, fontweight= "bold")
plt.ylabel("Score promedio")
plt.xticks(rotation=45)
plt.legend(title="Segmento")

plt.show()

Comentarios:
- El score promedio de los productos ha mostrado un compartamiento similar para las diferentes categorías de precios.
- Se observa un fuerte repunte del score a partir de Q3 2012.
- Los productos de precio "intermedio" son los más valorados por los usuarios en promedio.

In [ ]:
df_pivot.to_csv("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Estadística para Ciencia de Datos/Tareas/resultados/2.3pivot.csv", index=False, encoding="utf-8")
pivot_png.get_figure().savefig("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Estadística para Ciencia de Datos/Tareas/resultados/2.3pivot.png", dpi=300, bbox_inches="tight")

### 4. ¿Qué dicen los clientes insatisfechos vs. los satisfechos? 

In [ ]:
#Filtrado de clientes satisfechos

df_satisfechos = df_reseñas[df_reseñas["review_score"]>=4]

In [ ]:
df_satisfechos.describe(include="all")

In [ ]:
df_reseñas[["review_summary", "review_text"]].head()

In [ ]:
#Filtrado de clientes insatisfechos

df_insatisfechos = df_reseñas[df_reseñas["review_score"]<=2]

In [ ]:
#Me quedo con las 100 palabras más repetidas dentro de los satisfechos

textos = list(df_satisfechos["review_summary"])
texto_total = " ".join(textos)

texto_total = re.sub(r'[^a-z\s]', '', texto_total) #controlo caracteres raros, espacios, etc

lista_palabras = texto_total.split()

dic_satisfechos = {}

palabras_no_relevantes = set(STOPWORDS) #indico palabras no relevantes para filtrar

for palabra in lista_palabras:
    if palabra not in palabras_no_relevantes:
        dic_satisfechos[palabra] = dic_satisfechos.get(palabra, 0) + 1 #registro en diccionario palabra y cantidad de repeticiones

lista_final = list(dic_satisfechos.items())  #convierto a lista para poder ordenar
lista_final.sort(key=lambda x: x[1], reverse=True)

# Top 100
top_100_satisfechos = lista_final[:100] #capturo las 100 primeras palabras con más repeticiones

In [ ]:
textos = list(df_insatisfechos["review_summary"]) #idem pero para usuarios insatisfechos
texto_total = " ".join(textos)

texto_total = texto_total.lower()
texto_total = re.sub(r'[^a-z\s]', '', texto_total)

lista_palabras = texto_total.split()

dic_insatisfechos = {}

palabras_no_relevantes = set(STOPWORDS)

for palabra in lista_palabras:
    if palabra not in palabras_no_relevantes:
        dic_insatisfechos[palabra] = dic_insatisfechos.get(palabra, 0) + 1

# Ordenar
lista_final = list(dic_insatisfechos.items())
lista_final.sort(key=lambda x: x[1], reverse=True)

# Top 100
top_100_insatisfechos = lista_final[:100]

In [ ]:
print(top_100_satisfechos)

In [ ]:
#creamos listas para presentar información en un dataframe
palabras_satisfechos = []
cantidad_palabras_satisfechos = []
palabras_insatisfechos = []
cantidad_palabras_insatisfechos = []


for palabras in top_100_satisfechos:
    palabras_satisfechos.append(palabras[0])
    cantidad_palabras_satisfechos.append(palabras[1])

for palabras in top_100_insatisfechos:
    palabras_insatisfechos.append(palabras[0])
    cantidad_palabras_insatisfechos.append(palabras[1])


In [ ]:
df_palabras = pd.DataFrame({"satisfechos":palabras_satisfechos, "recuento_satisfechos":cantidad_palabras_satisfechos, "insatisfechos":palabras_insatisfechos, "recuento_insatisfechos": cantidad_palabras_insatisfechos})

In [ ]:
df_palabras

Comentarios: presentamos la información en un dataframe que muestra la frecuencia de palabras para ambos tipos de usuarios

In [ ]:
#usamos nube de palabras para presentar la información y visualizarlo de mejor manera, el dataframe nos pareció útil para responder la pregunta pero algo limitado
dic_top100 = dict(top_100_insatisfechos)

wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',colormap="Reds"
).generate_from_frequencies(dic_top100)

# Mostrar
plt.figure(figsize=(15, 15))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title("Top 100 palabras - Usuarios insatisfechos", fontsize=14, fontweight= "bold")
plt.show()

Comentarios:
- Se observa el uso de palabras negativas como "horrible", "bad", "waste", "useless", "disappointed etc. Creemos que hay una buena depuración de palabras no relevantes aunque no es perfecta. Igualmente observamos con claridad que las palabras son negativas, lo cual es esperable de un cliente que pone bajo score al producto

In [ ]:
dic_top100 = dict(top_100_satisfechos)

wordcloud2 = WordCloud(
    width=800,
    height=400,
    background_color='white', colormap="Blues"
).generate_from_frequencies(dic_top100)

# Mostrar
plt.figure(figsize=(15, 15))
plt.imshow(wordcloud2, interpolation='bilinear')
plt.axis('off')
plt.title("Top 100 palabras - Usuarios satisfechos", fontsize=14, fontweight= "bold")
plt.show()

Comentarios:
- Se observa el uso de palabras positivas como "love", "perfect", "nice", "great", awesome etc. Se nota un claro contraste con la nube de puntos anterior.El resultado es esperable considerando que son clientes satisfechos que pusieron buen score al producto.

In [ ]:
df_palabras.to_csv("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Programación para Ciencia de Datos/Tareas/resultados/2.4palabras.csv", index=False, encoding="utf-8")
wordcloud.to_file("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Programación para Ciencia de Datos/Tareas/resultados/2.4palabras_insatisfechos.png")
wordcloud2.to_file("C:/Users/nstan/Desktop/Maestría Ciencia de Datos/Programación para Ciencia de Datos/Tareas/resultados/2.4palabras_satisfechos.png")

## 5. ¿Quiénes son los reviewers más activos y qué tan representativos son?

In [ ]:
df_reseñas.columns

In [ ]:
top_usuarios = (df_reseñas.groupby("review_userId").agg(
    cantidad=("review_score", "count"),
    promedio_score=("review_score", "mean")
).sort_values("cantidad", ascending=False).head(5)
)

In [ ]:
promedio_top_5 = top_usuarios["promedio_score"].mean().round(2)

In [ ]:
promedio_score = df_reseñas["review_score"].mean()
print(f"Tabla top 5 usuarios con más reviews y su score promedio\n\n {top_usuarios.round(2)}")
print(f"\n\nEl score promedio de todos los usuarios identificables es {promedio_score: .2f}, mientras que el score promedio del grupo top 5 usuarios es {promedio_top_5}")

Comentarios: Vemos que el score promedio de estos usuarios está en todos los casos por encima del score promedio de toda la muestra.

In [ ]:
top_usuarios.to_csv("C:/Users/gonza/OneDrive/Escritorio/Master CD/Programacion DS/proyecto_final/2.5top_usuarios.csv", index=False, encoding="utf-8")

### 6. En esta parte vamos a plantear tres preguntas donde vamos a analizar la utilidad de las reseñas y como influye en las demas variables

### 6.1: ¿Las reseñas más útiles tienden a ser las más o menos críticas?

In [ ]:
df_reseñas["review_helpfulness"].unique()

In [ ]:
#Calculo ratio numérico para ayuda reseñas

def ratio(x):
    pares = x.split("/")
    numerador = int(pares[0])
    divisor = int(pares[1])
    if divisor !=0:
        ratio= numerador/divisor
    else:
        ratio=0

    return ratio


In [ ]:
df_reseñas["ratio_helpfulness"] = df_reseñas["review_helpfulness"].apply(ratio).round(2)

In [ ]:
df_reseñas["ratio_helpfulness"].sort_values(ascending=True).unique()

In [ ]:
reseña_por_critica=df_reseñas.groupby("review_score")["ratio_helpfulness"].mean().plot(
    figsize=(10,6),
    kind="bar",
    color="#2DCFF3",      
    edgecolor="black",
    linewidth=1.5
)

plt.xlabel("Review score")
plt.ylabel("Helpfulness ratio")
plt.xticks(rotation=0)
plt.title("¿Las reseñas más útiles son más o menos críticas?", fontsize=14, fontweight= "bold")
plt.show()

Comentario:
- Las reseñas más útiles parecen ser aquellas que son más críticas (de score más bajo)

### 6.2: ¿Los usuarios valoran mas las reseñas cuando los productos tienen un mayor valor de compra?

In [ ]:
reseñas_por_precio=df_reseñas.groupby("segmento_precio")["ratio_helpfulness"].mean().plot(
    figsize=(10,6),
    kind="bar",
    color="#2DCFF3",      
    edgecolor="black",
    linewidth=1.5)

plt.title("Helpfulness por segmento de precio", fontsize=14, fontweight= "bold")
plt.xlabel("Precios")
plt.xticks(rotation=0)
plt.ylabel("Helpfulness ratio")
plt.show()
plt.show()

Comentarios:
- Cuando analisamos por segmento de precios vemos que los usuarios valoran más las reviews en productos caros.

### 6.3:¿Una reseña mas larga significa mayor utilidad para los clientes? 

In [ ]:
df_reseñas["review_length"] = df_reseñas["review_text"].str.split().str.len()

In [ ]:
Utilidad_x_largo=plt.scatter(df_reseñas["review_length"], df_reseñas["ratio_helpfulness"], alpha=0.3)
plt.xlabel("Largo de la review")
plt.ylabel("Helpfulness")
plt.show()

In [ ]:
df_reseñas[["review_length", "ratio_helpfulness"]].corr()

#### observamos que reviews mas largas no se asocian a una mayor utilidad de las reseñas, ya que notamos una correlación de 0.2, lo cual es una correlación muy debil

In [ ]:
# guardamos gráficos y dataset final
reseña_por_critica.get_figure().savefig("C:/Users/gonza/OneDrive/Escritorio/Master CD/Programacion DS/proyecto_final/2.6.1reseña_por_critica.png", dpi=300, bbox_inches="tight")
reseñas_por_precio.get_figure().savefig("C:/Users/gonza/OneDrive/Escritorio/Master CD/Programacion DS/proyecto_final/2.6.2reseña_por_precio.png", dpi=300, bbox_inches="tight")
Utilidad_x_largo.get_figure().savefig("C:/Users/gonza/OneDrive/Escritorio/Master CD/Programacion DS/proyecto_final/2.6.3utilidad_x_largo.png", dpi=300, bbox_inches="tight")
df_reseñas.to_csv("C:/Users/gonza/OneDrive/Escritorio/Master CD/Programacion DS/proyecto_final/2.6reseñas_limpias.csv", index=False, encoding="utf-8")